[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garrygu/newegg-ai-workshop/blob/main/lv1-beginner-v2/Session_3_Image_Classification.ipynb)

# 🔍 Session 3 — Image Classification

Teach AI to **see** and recognize images! 🧠

In this session, you'll:
1. 📸 Understand how computers "see" images
2. 🧠 Build a neural network (CNN)
3. 🎯 Train it to classify images
4. 🎮 Create a custom classifier for your game

---

## 🎯 How Image Classification Works

```
📸 Input Image    →    🧠 Neural Network    →    🏷️ Label
  [cat photo]          (learns patterns)         "CAT"
```

The AI learns to extract **features** (edges, shapes, textures) and map them to categories!

---

## ⚙️ Part 1: Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using: {device}")

---

## 📊 Part 2: Load CIFAR-10 Dataset

CIFAR-10 contains 60,000 images in 10 categories:
✈️ airplane, 🚗 car, 🐦 bird, 🐱 cat, 🦌 deer, 🐕 dog, 🐸 frog, 🐴 horse, 🚢 ship, 🚚 truck

In [ ]:
# Image preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download datasets
print("📥 Downloading CIFAR-10...")
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Data loaders (batch processing)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

# Class names
classes = ('✈️plane', '🚗car', '🐦bird', '🐱cat', '🦌deer', 
           '🐕dog', '🐸frog', '🐴horse', '🚢ship', '🚚truck')

print(f"✅ Training: {len(trainset)} images")
print(f"✅ Testing: {len(testset)} images")

In [ ]:
# Visualize some training images
def show_images(images, labels, classes, n=8):
    """Display a grid of images with labels."""
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i in range(n):
        img = images[i].numpy().transpose((1, 2, 0))
        img = img * 0.5 + 0.5  # Unnormalize
        axes[i].imshow(img)
        axes[i].set_title(classes[labels[i]])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# Get sample batch
images, labels = next(iter(trainloader))
print("📸 Sample training images:")
show_images(images, labels, classes)

---

## 🧠 Part 3: Build a CNN

**CNN = Convolutional Neural Network**

CNNs are specially designed to understand images!

In [ ]:
class SimpleCNN(nn.Module):
    """A simple CNN for image classification."""
    
    def __init__(self):
        super().__init__()
        # Convolutional layers (feature extraction)
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)   # 32 filters
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)  # 64 filters
        self.pool = nn.MaxPool2d(2, 2)                 # Reduce size
        
        # Fully connected layers (classification)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)  # 10 classes
        
        # Activation and dropout
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        # Conv block 1
        x = self.pool(self.relu(self.conv1(x)))
        # Conv block 2
        x = self.pool(self.relu(self.conv2(x)))
        # Flatten
        x = x.view(-1, 64 * 8 * 8)
        # Fully connected
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Create model
model = SimpleCNN().to(device)
print("🧠 Model created!")
print(f"📊 Parameters: {sum(p.numel() for p in model.parameters()):,}")

<details>
<summary><strong>🔬 What Does Each Layer Do? (Click to Expand)</strong></summary>

| Layer | Purpose | Analogy |
|:--|:--|:--|
| **Conv2d** | Find patterns (edges, shapes) | Like using a magnifying glass |
| **MaxPool2d** | Shrink image, keep important info | Zooming out |
| **ReLU** | Add non-linearity | Makes learning possible |
| **Linear** | Make final decisions | Like voting |
| **Dropout** | Prevent overfitting | Forces robustness |

</details>

---

## 🏋️ Part 4: Train the Model

In [ ]:
# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training function
def train_epoch(model, loader, criterion, optimizer):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

In [ ]:
# Train the model (3 epochs for demo - increase for better results!)
EPOCHS = 3

print("🏋️ Training...\n")
for epoch in range(EPOCHS):
    loss, acc = train_epoch(model, trainloader, criterion, optimizer)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | Accuracy: {acc:.2f}%")

print("\n✅ Training complete!")

---

## 🧪 Part 5: Evaluate

In [ ]:
def evaluate(model, loader):
    """Evaluate model on test set."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return 100. * correct / total

test_acc = evaluate(model, testloader)
print(f"🎯 Test Accuracy: {test_acc:.2f}%")
print(f"💡 Random guessing would be: 10%")

In [ ]:
# Visualize predictions
def show_predictions(model, images, labels, classes):
    """Show model predictions on sample images."""
    model.eval()
    images_gpu = images.to(device)
    
    with torch.no_grad():
        outputs = model(images_gpu)
        _, predictions = outputs.max(1)
    
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < 8:
            img = images[i].numpy().transpose((1, 2, 0))
            img = img * 0.5 + 0.5
            ax.imshow(img)
            
            pred = classes[predictions[i]]
            true = classes[labels[i]]
            color = 'green' if predictions[i] == labels[i] else 'red'
            ax.set_title(f"Pred: {pred}\nTrue: {true}", color=color)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

# Get test samples
test_images, test_labels = next(iter(testloader))
print("📊 Model Predictions (green=correct, red=wrong):")
show_predictions(model, test_images, test_labels, classes)

---

## 💾 Part 6: Save Your Model

In [ ]:
import os
os.makedirs("models", exist_ok=True)

# Save model
torch.save(model.state_dict(), "models/classifier.pt")
print("💾 Model saved to models/classifier.pt")

In [ ]:
# How to load later:
def load_model(path):
    """Load a saved model."""
    model = SimpleCNN().to(device)
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

# loaded_model = load_model("models/classifier.pt")
# print("✅ Model loaded!")

---

## 🎮 Part 7: Use for Your Game (Preview)

Here's how you'll use classification in Session 5:

In [ ]:
def classify_image(model, image_tensor):
    """Classify a single image and return prediction with confidence."""
    model.eval()
    with torch.no_grad():
        if image_tensor.dim() == 3:
            image_tensor = image_tensor.unsqueeze(0)
        
        outputs = model(image_tensor.to(device))
        probs = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted = probs.max(1)
        
    return classes[predicted.item()], confidence.item()

# Example usage
sample_image = test_images[0]
label, confidence = classify_image(model, sample_image)
print(f"🔍 Prediction: {label}")
print(f"📊 Confidence: {confidence:.1%}")

---

## 🏆 Challenge Zone

### Challenge 1: Improve Accuracy

Try training for more epochs or adjusting the learning rate!

In [ ]:
# Train longer!
# for epoch in range(5):  # Try 5 or 10 epochs
#     loss, acc = train_epoch(model, trainloader, criterion, optimizer)
#     print(f"Epoch {epoch+1} | Accuracy: {acc:.2f}%")

### Challenge 2: Per-Class Accuracy

Which classes does the model struggle with?

In [ ]:
def per_class_accuracy(model, loader, classes):
    """Calculate accuracy for each class."""
    class_correct = [0] * 10
    class_total = [0] * 10
    
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            
            for i in range(len(labels)):
                label = labels[i].item()
                class_total[label] += 1
                if predicted[i] == labels[i]:
                    class_correct[label] += 1
    
    print("📊 Per-Class Accuracy:")
    for i, cls in enumerate(classes):
        acc = 100 * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
        bar = "█" * int(acc / 5) + "░" * (20 - int(acc / 5))
        print(f"{cls:8s} |{bar}| {acc:.1f}%")

per_class_accuracy(model, testloader, classes)

---

## 🔮 What's Coming Next!

In **Session 4: Chatbot, Sentiment & Voice**, you'll:
- Build a sentiment-aware chatbot
- Add voice input with Whisper
- Add voice output with TTS

---

## 🎯 Session 3 Wrap-Up

**You learned:**
- ✅ How CNNs extract image features
- ✅ The training loop (forward, loss, backward, optimize)
- ✅ How to evaluate model performance
- ✅ Saved a model for your game!

🎉 **You just trained your own AI!**